# PyPI → conda name mapping: empirical basis for the aggregator's resolution policy

`reroll` resolves PyPI project names to conda package names with three independent
mappers, each wrapping a different upstream data source:

| mapper | upstream data | probability semantics |
|---|---|---|
| `grayskull_mapper` | grayskull's hand-curated `config.yaml` exception table | always `1.0` (human-authored) |
| `conda_lock_mapper` | conda-forge autotick bot's `mappings/pypi` tables | `1.0` static override · `0.9` unambiguous · `0.6` ambiguous (HITS centrality) · `0.4` colliding spellings |
| `parselmouth_mapper` | parselmouth's `relations-v1` evidence table | `0.95 ×` name-axis `×` version-agreement (continuous) `×` contradiction factors; may return **several** candidates per name |

This report runs every name in this repo's PyPI corpus through all three mappers and
answers five questions, in order: how much do the sources overlap (§1); do the curated
sources ever disagree, and how good is conda-lock's own high-confidence tier (§2); does
parselmouth's probability actually predict agreement (§3); two candidate designs for the
last resolution rule, and how they compare end to end (§4a, §4b); and the algorithm we
adopt as a result, with its expected precision/accuracy (§5).

## The algorithm

Five rules, evaluated in order — the first one that fires wins:

```
resolve(pypi_name):
    candidates = independent answers from grayskull, conda_lock, parselmouth

    1. if grayskull has an answer:                       → take it
    2. elif conda_lock has a static-override answer (p == 1.0):
                                                           → take it
    3. elif >= 2 mappers propose the same conda name
       (vote over ALL candidates each mapper returned,
        not just each mapper's top pick):                → take it
    4. elif exactly one mapper still has an opinion:
           if that mapper is not parselmouth
              and its probability >= 0.9:                → take it
           elif that mapper is parselmouth
              and it returned exactly one candidate
              (at ANY probability):                       → take it
           else:                                          → defer
    5. else:                                              → defer
```

The key result behind rule 4 (§4a/§4b): for parselmouth, whether it returned a *single*
candidate predicts agreement with the other sources almost perfectly — regardless of its
own probability score. Raw probability, by contrast, predicts almost nothing (§3). So
rule 4 drops the probability floor for single-candidate parselmouth answers rather than
requiring both.

## Recommendation at a glance

Yield of each rule over the 19,663 names any mapper covers (§4b, §5):

| # | rule | names | share | precision proxy |
|---|---|---|---|---|
| 1 | **grayskull hit** → take it (curated) | 136 | 0.7% | 100% |
| 2 | **conda-lock static override** (p = 1.0) → take it (curated) | 14 | 0.1% | 100% |
| 3 | **≥ 2 mappers back the same conda name** → take it (vote over *all* candidates, not just each mapper's top) | 11,652 | 59.3% | 99.86% |
| 4 | **single source, p ≥ 0.9 — or any p for a single-candidate parselmouth answer** → take it | 7,561 | 38.5% | 99.99% |
| 5 | otherwise → **defer** | 300 | 1.5% | n/a |

**Composite precision ≈ 99.9%; coverage ≈ 98.5%; estimated overall correct-answer rate
≈ 98.4%** of the covered universe (§5).

Two alternative designs for rule 4 were evaluated side by side in §4b before settling on
the algorithm above:

- **A more conservative variant** that also required p ≥ 0.9 for single-candidate
  parselmouth answers yields 7,038 names / a 4.2% defer rate — 2.7 points less coverage,
  at no precision benefit, because single-candidacy alone (not probability) is what
  predicts agreement.
- **A more aggressive variant** that also accepts multi-candidate p ≥ 0.9 answers
  recovers 281 more names but at a ~9% error rate concentrated in that one bucket.

Headline findings:

- **Coverage is lopsided.** Parselmouth answers 19,611 of the 19,663 covered names;
  conda-lock is nearly a strict subset of it; grayskull adds no unique names (§1).
- **The curated sources never disagree** (0/84 overall, 0/17 within conda-lock's static
  tier specifically) — either can be trusted outright once it fires (§2A).
- **Conda-lock's non-static high tier holds up well on its own:** 99.87% agreement with
  parselmouth, no curated override involved (§2B).
- **Probability alone barely predicts agreement.** Every confidence tier — on both
  mappers — agrees at 98-100%, except conda-lock's own lowest "colliding spellings" tier
  (63.6%, but only 11 names). Confidence is not a useful risk signal here (§3).
- **"Multiple candidates exist" is a much better risk signal than probability, and it
  makes the probability floor redundant for single-candidate answers.** Gating on
  candidate count alone matches gating on probability-and-candidate-count for precision,
  while recovering 523 more names — a genuine win, not a trade (§4a, §4b).
- **A more aggressive variant was tested and rejected.** Also accepting multi-candidate,
  p ≥ 0.9 answers closes the last 1.4 points of coverage at a 10x increase in error,
  concentrated entirely in that bucket (§4b).

Data snapshot: universe is every distinct, canonicalized `project.name` in this repo's
`data/v.db` PyPI crawl (865,839 names); results are cached to
`data/name_mapper_candidates.parquet`; the parselmouth relations snapshot is fetched at
notebook runtime and regenerated upstream hourly.

In [1]:
import sqlite3
import time
from pathlib import Path

import polars as pl
from IPython.display import display
from packaging.utils import canonicalize_name

from reroll.name_mapping import Candidate
from reroll.grayskull_mapper import grayskull_mapper
from reroll.conda_lock_mapper import conda_lock_mapper
from reroll.parselmouth_mapper import open_parselmouth_database, parselmouth_mapper

# Kernel runs in this repo's uv environment (pyproject auto-detected), but its cwd is
# this notebook's own directory (notebooks/), not the repo root -- so locate the root by
# walking up to the directory that actually holds pyproject.toml, rather than assuming
# Path.cwd() already is it (that assumption silently pointed DATA at notebooks/data,
# where sqlite3 happily auto-creates an empty v.db instead of erroring).
def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise SystemExit(f"could not find pyproject.toml above {start}")


ROOT = _find_repo_root(Path.cwd())
DATA = ROOT / "data"
V_DB = DATA / "v.db"  # this repo's PyPI corpus; see the Makefile
PARSELMOUTH_DB = DATA / "parselmouth/mapping.sqlite3"

print("cwd:", ROOT)
import reroll

print("reroll from:", reroll.__file__)

cwd: /Users/anil/code/reroll-data
reroll from: /Users/anil/code/reroll/src/reroll/__init__.py


---
## Data and method

**Universe.** Every distinct `project.name` in this repo's `v.db` PyPI index crawl,
canonicalized (`map_name` canonicalizes before dispatch, so the mappers see canonical
names).

**Independent opinions.** Each mapper is called with an empty candidate sequence —
exactly as if it were first in a chain — so every mapper's answer is its own, uninformed
by the others. A mapper returning the input unchanged means "no opinion"; anything it
appends is that mapper's candidate(s) for the name. All results land in one long-format
dataframe (`pypi_name, mapper, conda_name, probability`).

**Mapper construction** goes through reroll's public API, which handles its own data
acquisition:

- `grayskull_mapper()` — reads the `config.yaml` bundled with the installed grayskull. Offline.
- `conda_lock_mapper()` — fetches the autotick bot's three mapping tables through
  conda-lock's ETag-conditional cache (network on first run, cached afterwards).
- `parselmouth_mapper(conn)` — `open_parselmouth_database` downloads parselmouth's
  `relations.jsonl.gz` (ETag-conditional) and rebuilds the local sqlite evidence db only
  when upstream changed. Its probability is `0.95 ×` name-axis `×` a continuous
  version-agreement factor (linearly interpolated by the agreeing share of informative
  versions, not a step function) `×` contradiction factors.

**Sections, in order:** coverage overlap (§1) → curated agreement + conda-lock's own
high tier (§2) → does parselmouth's probability predict agreement (§3) → two candidate
designs for rule 4 from a multiplicity signal (§4a) → comparing them end to end (§4b) →
the adopted algorithm and expected precision/accuracy (§5).

In [2]:
t = time.time()
gsk = grayskull_mapper()
print(f"grayskull mapper built in {time.time() - t:.1f}s")

t = time.time()
clk = conda_lock_mapper()
print(f"conda-lock mapper built in {time.time() - t:.1f}s (includes cached downloads)")

t = time.time()
pconn = open_parselmouth_database(PARSELMOUTH_DB)
psm = parselmouth_mapper(pconn)
print(f"parselmouth db opened in {time.time() - t:.1f}s")
print(
    "pypi_conda_mapping rows:",
    pconn.execute("SELECT COUNT(*) FROM pypi_conda_mapping").fetchone()[0],
)

grayskull mapper built in 0.0s
conda-lock mapper built in 0.8s (includes cached downloads)
parselmouth db opened in 11.1s
pypi_conda_mapping rows: 21098


### Running the universe

The full run maps ~866k names through three mappers and is cached to
`data/name_mapper_candidates.parquet`; re-running this notebook loads the cache instead
of re-computing. Delete the parquet file to regenerate against fresher upstream data.


In [3]:
v = sqlite3.connect(V_DB)
names = sorted(
    {canonicalize_name(r[0]) for r in v.execute("SELECT DISTINCT name FROM project")}
)
v.close()
print(f"{len(names):,} canonicalized PyPI names")

CACHE = DATA / "name_mapper_candidates.parquet"
if CACHE.exists():
    candidates = pl.read_parquet(CACHE)
    print(f"loaded {candidates.height:,} cached candidates from {CACHE}")
else:
    t = time.time()
    rows = []
    for name in names:
        for mapper_name, m in (
            ("grayskull", gsk),
            ("conda_lock", clk),
            ("parselmouth", psm),
        ):
            for c in m(name, ()):
                rows.append((name, mapper_name, c.conda_name, c.probability))
    candidates = pl.DataFrame(
        rows, schema=["pypi_name", "mapper", "conda_name", "probability"], orient="row"
    )
    candidates.write_parquet(CACHE)
    print(f"{time.time() - t:.1f}s, {candidates.height:,} candidates -> {CACHE}")

candidates = candidates.with_columns(
    pl.col("conda_name")
    .map_elements(canonicalize_name, return_dtype=pl.String)
    .alias("conda_canon")
)

865,839 canonicalized PyPI names
loaded 32,187 cached candidates from /Users/anil/code/reroll-data/data/name_mapper_candidates.parquet


In [4]:
# Shared objects reused by every section below.

# Each mapper's single best (highest-probability) candidate per name.
top = candidates.sort("probability", descending=True).unique(
    subset=["pypi_name", "mapper"], keep="first"
)

sets = {
    m: set(candidates.filter(pl.col("mapper") == m)["pypi_name"])
    for m in ("grayskull", "conda_lock", "parselmouth")
}

# Per-name summary: which mapper(s) cover it, its best probability, how many
# parselmouth candidates it has.
per_name = candidates.group_by("pypi_name").agg(
    pl.col("mapper").unique().sort().str.join("+").alias("sources"),
    pl.col("probability").max().alias("best_p"),
)
psm_ncand = (
    candidates.filter(pl.col("mapper") == "parselmouth")
    .group_by("pypi_name")
    .agg(pl.len().alias("n_cand"))
)


def pair_frame(a, b):
    """Top-candidate comparison between two mappers, over the names both cover."""
    ca = top.filter(pl.col("mapper") == a).select(
        "pypi_name",
        conda_a=pl.col("conda_canon"),
        p_a=pl.col("probability"),
        raw_a=pl.col("conda_name"),
    )
    cb = top.filter(pl.col("mapper") == b).select(
        "pypi_name",
        conda_b=pl.col("conda_canon"),
        p_b=pl.col("probability"),
        raw_b=pl.col("conda_name"),
    )
    return ca.join(cb, on="pypi_name").with_columns(
        (pl.col("conda_a") == pl.col("conda_b")).alias("agree")
    )


cl_ps = pair_frame("conda_lock", "parselmouth")  # conda-lock (a) vs parselmouth (b)
g_cl = pair_frame("grayskull", "conda_lock")
g_ps = pair_frame("grayskull", "parselmouth")

# Names a human already had to hand-correct in a curated source: grayskull's whole
# table, or conda-lock's p=1.0 static-override tier. Every §2-onward question about
# "how good is the evidence-based signal" excludes these, because scoring an
# automatic guess against a row that exists *because the automatic guess was wrong*
# measures how often humans intervene, not how good the confidence signal is.
conda_lock_static_names = set(
    top.filter((pl.col("mapper") == "conda_lock") & (pl.col("probability") == 1.0))[
        "pypi_name"
    ]
)
outlier_names = sets["grayskull"] | conda_lock_static_names
print(
    f"grayskull-curated: {len(sets['grayskull'])}, conda-lock static: {len(conda_lock_static_names)}, union: {len(outlier_names)}"
)

# All (pypi_name, conda_name) pairs any mapper proposed, with how many distinct
# mappers back each pair -- the basis for the "≥2 mappers agree" vote rule (§4b, §5).
votes = (
    candidates.group_by(["pypi_name", "conda_canon"])
    .agg(pl.col("mapper").unique().sort().alias("mappers"), pl.col("probability").max().alias("max_p"))
    .with_columns(pl.col("mappers").list.len().alias("n_mappers"))
)

grayskull-curated: 136, conda-lock static: 31, union: 150


---
## 1. Coverage overlap: what can each source answer, and how much do they share?

*Question: how much of PyPI does each mapper cover, and how much overlap exists between
the three sources?*


In [5]:
by_mapper = (
    candidates.group_by("mapper")
    .agg(pl.col("pypi_name").n_unique().alias("names_covered"))
    .with_columns(
        (pl.col("names_covered") / len(names) * 100).round(3).alias("pct_of_pypi")
    )
    .sort("names_covered", descending=True)
)
display(by_mapper)

overlap_rows = []
for a, b in (
    ("conda_lock", "parselmouth"),
    ("grayskull", "conda_lock"),
    ("grayskull", "parselmouth"),
):
    inter = len(sets[a] & sets[b])
    overlap_rows.append((f"{a} ∩ {b}", inter, round(inter / len(sets[a] | sets[b]), 4)))
overlap_rows.append(("all three ∩", len(set.intersection(*sets.values())), None))
overlap_rows.append(
    ("any mapper (universe covered)", len(set.union(*sets.values())), None)
)
display(
    pl.DataFrame(overlap_rows, schema=["overlap", "names", "jaccard"], orient="row")
)

print("names covered by exactly N mappers:")
display(
    per_name.with_columns(
        pl.col("sources").str.count_matches(r"\+").add(1).alias("n_sources")
    )
    .group_by("n_sources")
    .agg(pl.len().alias("names"))
    .sort("n_sources")
)

mapper,names_covered,pct_of_pypi
str,u32,f64
"""parselmouth""",19611,2.265
"""conda_lock""",11802,1.363
"""grayskull""",136,0.016


overlap,names,jaccard
str,i64,f64
"""conda_lock ∩ parselmouth""",11750,0.5976
"""grayskull ∩ conda_lock""",84,0.0071
"""grayskull ∩ parselmouth""",135,0.0069
"""all three ∩""",83,null
"""any mapper (universe covered)""",19663,null


names covered by exactly N mappers:


n_sources,names
u32,u32
1,7860
2,11720
3,83


**Finding:**

- **Parselmouth is the coverage backbone** (19,611 names, 2.27% of PyPI); conda-lock
  (11,802) is nearly a strict subset of it (only 51 conda-lock-only names); grayskull
  (136) adds no unique names of its own.
- **Union coverage is 19,663 names.** Everything else in PyPI's 865,839 names has no
  conda-forge counterpart and falls through to the caller's identity/default policy.
- **11,720 names (60% of covered names) are backed by ≥ 2 mappers** — these get the
  corroboration treatment in §3–§4a. The remaining 7,860 are single-source (7,809
  parselmouth-only, 51 conda-lock-only) — confidence thresholds are the only gate there.


---
## 2. Do the curated sources ever disagree, and how good is conda-lock's own high-confidence tier?

*Question A: for names with a `1.0` static mapping (human-curated — grayskull's whole
table, or conda-lock's static-override tier), does grayskull ever disagree with
conda-lock?*

*Question B: for conda-lock's high-probability tier (`0.90 ≤ p < 1.0` — its "unambiguous"
tier, deliberately excluding the `1.0` hand-curated tier), how often does parselmouth
agree?*


In [6]:
print("A) grayskull x conda_lock, all shared names (any confidence):")
print(f"   shared: {g_cl.height}, disagree: {g_cl.filter(~pl.col('agree')).height}")

print(f"\nA) restricted to conda-lock's 1.0 static-override tier specifically:")
g_cl_static = g_cl.filter(pl.col("pypi_name").is_in(list(conda_lock_static_names)))
print(
    f"   of {len(conda_lock_static_names)} static-override names, grayskull also covers {g_cl_static.height}"
)
print(
    f"   of those, grayskull agrees: {g_cl_static.filter(pl.col('agree')).height}/{g_cl_static.height}"
)

print("\nB) conda-lock's high-probability tier (0.90 <= p < 1.0) vs parselmouth:")
cl_high = cl_ps.filter((pl.col("p_a") >= 0.9) & (pl.col("p_a") < 1.0))
print(
    f"   shared names: {cl_high.height}, agree: {cl_high['agree'].sum()} ({cl_high['agree'].mean():.4%})"
)
display(cl_high.filter(~pl.col("agree")).select("pypi_name", "raw_a", "raw_b", "p_b"))

A) grayskull x conda_lock, all shared names (any confidence):
   shared: 84, disagree: 0

A) restricted to conda-lock's 1.0 static-override tier specifically:
   of 31 static-override names, grayskull also covers 17
   of those, grayskull agrees: 17/17

B) conda-lock's high-probability tier (0.90 <= p < 1.0) vs parselmouth:
   shared names: 11240, agree: 11225 (99.8665%)


pypi_name,raw_a,raw_b,p_b
str,str,str,f64
"""qtconsole""","""qtconsole-base""","""qtconsole""",0.95
"""mdahole2""","""mdahole2""","""mdahole2-base""",0.76
"""lit""","""lit-nlp""","""lit""",0.8856
"""ribs""","""pyribs-base""","""pyribs""",0.95
"""pyhyperscattering""","""pyhyperscattering-base""","""pyhyperscattering""",0.95
…,…,…,…
"""pandera""","""pandera-core""","""pandera""",0.95
"""sagemaker""","""sagemaker-python-sdk""","""sagemaker""",0.95
"""pyam-iamc""","""pyam""","""pyam-iamc""",0.95


**Finding:**

- **A) The two curated sources never disagree.** 0/84 conflicts overall, and 0/17
  specifically within conda-lock's 1.0 static-override tier where grayskull also has an
  opinion. This is expected — both are hand-authored exception tables, not independent
  guesses — but it confirms neither curated source is silently wrong relative to the
  other, so either can be trusted outright once it fires.
- **B) Conda-lock's non-static "unambiguous" tier (0.90 ≤ p < 1.0) agrees with parselmouth
  99.87% of the time** (11,225/11,240 shared names). This tier is not hand-curated — it's
  conda-lock's algorithmic top confidence bucket — and it still checks out almost
  perfectly against a second, independently-built source. The 15 residual disagreements
  are genuine split-package cases (e.g. `cockroachdb` → `cockroachdb-python` vs.
  `cockroachdb`), not curation gaps.


---
## 3. Does parselmouth's probability actually predict agreement with conda-lock?

*Question: excluding every name with a 1.0 static mapping in grayskull or conda-lock
(§2's curated tier), tranche the remaining conda-lock ∩ parselmouth overlap by conda-lock
probability (high/medium/low = its 0.9/0.6/0.4 tiers) crossed with parselmouth probability
(high ≥ 0.9, medium 0.5–0.9, low < 0.5). Is there a correlation between either side's
confidence and whether they agree?*


In [7]:
main3 = cl_ps.filter(~pl.col("pypi_name").is_in(list(outlier_names)))
print(
    f"main analysis set (conda_lock x parselmouth, curated 1.0 names excluded): {main3.height:,}"
)

lock_order = ["low (0.4)", "medium (0.6)", "high (0.9)"]
psm_order = ["low (<0.5)", "medium (0.5-0.9)", "high (>=0.9)"]

tiered = main3.with_columns(
    pl.col("p_a")
    .replace_strict({0.4: "low (0.4)", 0.6: "medium (0.6)", 0.9: "high (0.9)"})
    .cast(pl.Enum(lock_order))
    .alias("lock_tier"),
    pl.when(pl.col("p_b") >= 0.9)
    .then(pl.lit("high (>=0.9)"))
    .when(pl.col("p_b") >= 0.5)
    .then(pl.lit("medium (0.5-0.9)"))
    .otherwise(pl.lit("low (<0.5)"))
    .cast(pl.Enum(psm_order))
    .alias("psm_tier"),
)

print("\ncross-tab: conda-lock tier x parselmouth tier -> agreement rate")
display(
    tiered.group_by(["lock_tier", "psm_tier"])
    .agg(
        pl.len().alias("n"),
        pl.col("agree").sum().alias("agree_n"),
        pl.col("agree").mean().round(4).alias("agree_rate"),
    )
    .sort(["lock_tier", "psm_tier"])
)

print("\nmarginal: by conda-lock tier alone")
display(
    tiered.group_by("lock_tier")
    .agg(pl.len().alias("n"), pl.col("agree").mean().round(4).alias("agree_rate"))
    .sort("lock_tier")
)
print("\nmarginal: by parselmouth tier alone")
display(
    tiered.group_by("psm_tier")
    .agg(pl.len().alias("n"), pl.col("agree").mean().round(4).alias("agree_rate"))
    .sort("psm_tier")
)

main analysis set (conda_lock x parselmouth, curated 1.0 names excluded): 11,653

cross-tab: conda-lock tier x parselmouth tier -> agreement rate



marginal: by conda-lock tier alone


lock_tier,psm_tier,n,agree_n,agree_rate
enum,enum,u32,u32,f64
"""low (0.4)""","""medium (0.5-0.9)""",1,1,1.0
"""low (0.4)""","""high (>=0.9)""",10,6,0.6
"""medium (0.6)""","""low (<0.5)""",1,1,1.0
"""medium (0.6)""","""medium (0.5-0.9)""",4,4,1.0
"""medium (0.6)""","""high (>=0.9)""",462,462,1.0
"""high (0.9)""","""low (<0.5)""",34,34,1.0
"""high (0.9)""","""medium (0.5-0.9)""",107,105,0.9813
"""high (0.9)""","""high (>=0.9)""",11034,11024,0.9991



marginal: by parselmouth tier alone


lock_tier,n,agree_rate
enum,u32,f64
"""low (0.4)""",11,0.6364
"""medium (0.6)""",467,1.0
"""high (0.9)""",11175,0.9989


psm_tier,n,agree_rate
enum,u32,f64
"""low (<0.5)""",35,1.0
"""medium (0.5-0.9)""",112,0.9821
"""high (>=0.9)""",11506,0.9988


**Finding: essentially no correlation — agreement is flat at ~98-100% across every tier
combination, with one sharp exception.**

- **Parselmouth's own confidence tier doesn't predict agreement at all.** Low (< 0.5,
  n=35): 100%. Medium (0.5–0.9, n=112): 98.2%. High (≥ 0.9, n=11,506): 99.9%. A name
  parselmouth scores at 0.3 is corroborated exactly as often as one it scores at 0.95.
- **Conda-lock's tier is almost as flat** — medium (0.6) is 100%, high (0.9) is 99.9% —
  **except its own lowest "colliding spellings" tier (0.4, n=11)**, which drops to 63.6%.
  That is the one real risk signal in this table, and it belongs to conda-lock, not
  parselmouth.
- **The 2×2 high/high cell carries 95% of the data** (11,034/11,653) and agrees 99.9% of
  the time — reassuring for the bulk case, but it means the cross-tab can't actually test
  whether confidence discriminates risk, because there's almost no data in the
  low-confidence cells to compare against. Neither side's probability is a useful risk
  signal here; something else must be driving the residual ~15 disagreements (§4a).


---
## 4a. Two proposals for rule 4: does a multiplicity signal improve on raw probability?

*Question: §3 found probability nearly useless as a risk signal. If we add an extra
signal — parselmouth returned more than one candidate for this name — and use it to
artificially disqualify a name from the "accept outright" tier even when its top
probability is ≥ 0.9, does that materially improve precision and accuracy versus the
plain probability-only rule? And once multiplicity is in play, does raw probability add
anything further?*

Rule under test, both evaluated on `main3` (curated 1.0 names already excluded):
- **baseline** — accept iff raw parselmouth probability ≥ 0.9
- **multiplicity-adjusted** — accept iff probability ≥ 0.9 **and** parselmouth returned
  exactly one candidate

"Ground truth" is agreement with conda-lock's independent answer (the only check
available); precision = P(agree | accepted), accuracy = P(correct decision) over all of
`main3`, treating an accept-and-agree or reject-and-disagree as correct.

In [8]:
m4 = main3.join(psm_ncand, on="pypi_name")


def confusion(frame, predicted_col):
    tp = frame.filter(pl.col(predicted_col) & pl.col("agree")).height
    fp = frame.filter(pl.col(predicted_col) & ~pl.col("agree")).height
    fn = frame.filter(~pl.col(predicted_col) & pl.col("agree")).height
    tn = frame.filter(~pl.col(predicted_col) & ~pl.col("agree")).height
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    accuracy = (tp + tn) / frame.height
    return dict(
        n_accept=tp + fp,
        tp=tp,
        fp=fp,
        fn=fn,
        tn=tn,
        precision=round(precision, 5),
        accuracy=round(accuracy, 5),
    )


m4 = m4.with_columns(
    (pl.col("p_b") >= 0.9).alias("accept_baseline"),
    ((pl.col("p_b") >= 0.9) & (pl.col("n_cand") == 1)).alias(
        "accept_multiplicity_adjusted"
    ),
)

print("baseline rule (accept iff raw parselmouth p >= 0.9):")
display(pl.DataFrame([confusion(m4, "accept_baseline")]))
print("\nmultiplicity-adjusted rule (accept iff p >= 0.9 AND exactly one candidate):")
display(pl.DataFrame([confusion(m4, "accept_multiplicity_adjusted")]))

print(
    "\nwhy: disagreement rate of the >= 0.9 raw-accept bucket, split by candidate count:"
)
display(
    m4.filter(pl.col("accept_baseline"))
    .group_by((pl.col("n_cand") > 1).alias("multi_candidate"))
    .agg(
        pl.len().alias("n"),
        (1 - pl.col("agree").mean()).round(4).alias("disagree_rate"),
    )
)

baseline rule (accept iff raw parselmouth p >= 0.9):


n_accept,tp,fp,fn,tn,precision,accuracy
i64,i64,i64,i64,i64,f64,f64
11506,11492,14,145,2,0.99878,0.98636



multiplicity-adjusted rule (accept iff p >= 0.9 AND exactly one candidate):



why: disagreement rate of the >= 0.9 raw-accept bucket, split by candidate count:


n_accept,tp,fp,fn,tn,precision,accuracy
i64,i64,i64,i64,i64,f64,f64
11349,11349,0,288,16,1.0,0.97529



why: disagreement rate of the >= 0.9 raw-accept bucket, split by candidate count:


multi_candidate,n,disagree_rate
bool,u32,f64
false,11349,0.0
true,157,0.0892


**Finding: yes for precision, no for accuracy — it's a genuine precision/coverage
trade, not a free win.**

| rule | accepted | precision | accuracy |
|---|---|---|---|
| baseline (p ≥ 0.9) | 11,506 | 99.878% | 98.636% |
| + multiplicity flag (p ≥ 0.9 and 1 candidate) | 11,349 | **100.000%** | 97.529% |

- **Precision improves to perfect** in the checkable sample: every one of the 14 false
  accepts under the baseline rule comes from a multi-candidate name (multi-candidate
  disagreement rate 8.9% vs. 0.0% for single-candidate), so flagging multiplicity
  eliminates all 14 observed false accepts.
- **Accuracy gets slightly worse** (98.64% → 97.53%), because the flag is binary and
  indiscriminate: it also demotes the 143 multi-candidate names that were actually
  *correct* (157 total multi-candidate accepts − 14 wrong = 143 right ones lost along with
  the 14 wrong ones). Those 143 aren't lost from the aggregator overall — they fall
  through to corroboration/vote rules (§5) — but judged as a standalone probability rule,
  raw accuracy drops.
- **Net verdict: worth adding as a gate, not as a replacement for probability** — *but*
  is probability doing any work once multiplicity is already accounted for? The next
  cell checks whether single-candidate names need a probability floor at all.

In [9]:
# Is single-candidacy alone -- regardless of raw probability -- still reliable, on the
# checkable subset (main3 ∩ conda-lock)? If so, the probability floor in the
# multiplicity-gated rule above is screening for the wrong thing: once a name is
# single-candidate, its own score shouldn't matter.
single_only = m4.filter(pl.col("n_cand") == 1)
multi_high = m4.filter((pl.col("n_cand") > 1) & (pl.col("p_b") >= 0.9))
print(
    f"checkable single-candidate names, ANY probability: {single_only.height:,}, agreement {single_only['agree'].mean():.4%}"
)
display(
    single_only.with_columns(
        pl.col("p_b")
        .cut(
            [0.3, 0.5, 0.7, 0.9],
            labels=["<0.3", "0.3-0.5", "0.5-0.7", "0.7-0.9", ">=0.9"],
        )
        .alias("p_band")
    )
    .group_by("p_band")
    .agg(pl.len().alias("n"), pl.col("agree").mean().round(4).alias("agree_rate"))
    .sort("p_band")
)
print(
    f"\nfor comparison, the bucket already gated OUT above -- checkable multi-candidate, p>=0.9: {multi_high.height}, agreement {multi_high['agree'].mean():.4%}"
)

checkable single-candidate names, ANY probability: 11,494, agreement 99.9913%



for comparison, the bucket already gated OUT above -- checkable multi-candidate, p>=0.9: 157, agreement 91.0828%


p_band,n,agree_rate
enum,u32,f64
"""<0.3""",5,1.0
"""0.3-0.5""",30,1.0
"""0.5-0.7""",18,1.0
"""0.7-0.9""",92,0.9891
""">=0.9""",11349,1.0



for comparison, the bucket already gated OUT above -- checkable multi-candidate, p>=0.9: 157, agreement 91.0828%


**Finding: single-candidacy alone predicts agreement almost perfectly, regardless of raw
probability — the floor is screening for the wrong thing.**

11,493/11,494 checkable single-candidate answers agree with conda-lock, including the 5
that score below 0.3 — while the multi-candidate, p ≥ 0.9 bucket gated out above still
carries an 8.9% error rate (91.08% agreement). Probability isn't adding discrimination
once candidate count is known; it's multiplicity, full stop.

**Two candidate designs for rule 4, carried forward as complete algorithms and compared
head-to-head in §4b:**

- **Probability-gated** — accept a single remaining source iff its probability is
  ≥ 0.9, **and**, if the source is parselmouth, it returned exactly one candidate.
- **Multiplicity-gated (Policy A)** — accept a single remaining source iff
  (non-parselmouth) its probability is ≥ 0.9, **or** (parselmouth) it returned exactly
  one candidate, at *any* probability.

§4b runs both as full 5-rule algorithms — not just rule 4 in isolation — and compares
coverage, precision, and overall correctness end to end.

---
## 4b. Comparing the two policies end to end

*Question: §4a's two rule-4 designs imply two different complete algorithms. Run both as
full 5-rule policies — how do coverage, precision, and overall correctness compare, and
does Policy A's extra coverage come at a hidden cost anywhere?*

In [10]:
def simulate(rule4_predicate):
    """Run the full 5-rule policy with a given rule-4 predicate; return the yield-by-rule
    dict and the set of names left unresolved (for inspection)."""
    resolved_v, remaining_v = {}, per_name.select("pypi_name")

    def take_v(mask_frame):
        nonlocal remaining_v
        hit = mask_frame.select("pypi_name").unique()
        remaining_v = remaining_v.filter(
            ~pl.col("pypi_name").is_in(hit["pypi_name"].implode())
        )
        return hit.height

    resolved_v["1_grayskull"] = take_v(
        votes.filter(
            pl.col("mappers").list.contains("grayskull")
            & pl.col("pypi_name").is_in(remaining_v["pypi_name"].implode())
        )
    )
    resolved_v["2_lock_static"] = take_v(
        votes.filter(
            pl.col("mappers").list.contains("conda_lock")
            & (pl.col("max_p") == 1.0)
            & pl.col("pypi_name").is_in(remaining_v["pypi_name"].implode())
        )
    )
    resolved_v["3_two_source_vote"] = take_v(
        votes.filter(
            (pl.col("n_mappers") >= 2)
            & pl.col("pypi_name").is_in(remaining_v["pypi_name"].implode())
        )
    )
    pool = (
        per_name.join(psm_ncand, on="pypi_name", how="left")
        .with_columns(pl.col("n_cand").fill_null(0))
        .filter(pl.col("pypi_name").is_in(remaining_v["pypi_name"].implode()))
    )
    resolved_v["4_single_source"] = take_v(pool.filter(rule4_predicate))
    resolved_v["5_deferred"] = remaining_v.height
    return resolved_v, remaining_v


# the two rule-4 designs from §4a, as full predicates over per_name + psm_ncand
pred_current = ((pl.col("sources") != "parselmouth") & (pl.col("best_p") >= 0.9)) | (
    (pl.col("sources") == "parselmouth")
    & (pl.col("n_cand") == 1)
    & (pl.col("best_p") >= 0.9)
)
pred_A = ((pl.col("sources") != "parselmouth") & (pl.col("best_p") >= 0.9)) | (
    (pl.col("sources") == "parselmouth") & (pl.col("n_cand") == 1)
)
# a third, more aggressive option -- considered and rejected below
pred_B = pred_A | (
    (pl.col("sources") == "parselmouth")
    & (pl.col("n_cand") > 1)
    & (pl.col("best_p") >= 0.9)
)

resolved, remaining = simulate(pred_current)  # "current algorithm"
resolved_A, remaining_A = simulate(pred_A)  # "Policy A"
resolved_B, remaining_B = simulate(pred_B)  # rejected aggressive option

precision_3_vote = main3[
    "agree"
].mean()  # §3: non-curated conda_lock x parselmouth agreement
precision_single_any_p = single_only[
    "agree"
].mean()  # §4a: parselmouth, n_cand==1, any p
precision_multi_high = multi_high["agree"].mean()  # §4a: parselmouth, n_cand>1, p>=0.9

for label, r in (
    ("current algorithm", resolved),
    ("Policy A", resolved_A),
    ("rejected aggressive option", resolved_B),
):
    tot = sum(r.values())
    print(
        f"{label}: decided={tot - r['5_deferred']:,}, deferred={r['5_deferred']} ({r['5_deferred'] / tot:.2%})"
    )

current algorithm: decided=18,840, deferred=823 (4.19%)
Policy A: decided=19,363, deferred=300 (1.53%)
rejected aggressive option: decided=19,644, deferred=19 (0.10%)


In [11]:
deferred = (
    remaining.join(per_name, on="pypi_name")
    .join(psm_ncand, on="pypi_name", how="left")
    .with_columns(pl.col("n_cand").fill_null(0))
)
print(f"deferred under the current algorithm: {deferred.height}")

print("\nby source combination:")
display(
    deferred.group_by("sources").agg(
        pl.len().alias("n"), pl.col("best_p").median().round(3).alias("median_best_p")
    )
)

print("\nthe 820 parselmouth-only deferred names, by why they missed rule 4:")
display(
    deferred.filter(pl.col("sources") == "parselmouth")
    .with_columns(
        pl.col("best_p").cut([0.9], labels=["p < 0.9", "p >= 0.9"]).alias("p_band"),
        (pl.col("n_cand") > 1).alias("multi_candidate"),
    )
    .group_by(["p_band", "multi_candidate"])
    .agg(pl.len().alias("n"))
    .sort(["p_band", "multi_candidate"])
)

deferred under the current algorithm: 823

by source combination:



the 820 parselmouth-only deferred names, by why they missed rule 4:


sources,n,median_best_p
str,u32,f64
"""parselmouth""",820,0.776
"""conda_lock""",3,0.6



the 820 parselmouth-only deferred names, by why they missed rule 4:


p_band,multi_candidate,n
enum,bool,u32
"""p < 0.9""",false,523
"""p < 0.9""",true,16
"""p >= 0.9""",true,281


In [12]:
def composite(resolved_v, rule4_parts):
    """Weight each rule's yield by a precision proxy, combine with coverage to get an
    expected correct-answer rate over the whole covered universe."""
    parts = [
        (resolved_v["1_grayskull"], 1.0),
        (resolved_v["2_lock_static"], 1.0),
        (resolved_v["3_two_source_vote"], precision_3_vote),
        *rule4_parts,
    ]
    n_decided = sum(n for n, _ in parts)
    n_total = n_decided + resolved_v["5_deferred"]
    prec = sum(n * p for n, p in parts) / n_decided
    cov = n_decided / n_total
    return dict(
        decided=n_decided,
        deferred=resolved_v["5_deferred"],
        coverage=round(cov, 4),
        precision=round(prec, 5),
        overall=round(prec * cov, 4),
    )


rows = [
    (
        "current algorithm (multiplicity-gated)",
        composite(resolved, [(resolved["4_single_source"], 1.0)]),
    ),
    (
        "Policy A (drop prob floor for single-candidate)",
        composite(
            resolved_A, [(resolved_A["4_single_source"], precision_single_any_p)]
        ),
    ),
    (
        "(rejected) Policy A + accept multi-candidate p>=0.9 anyway",
        composite(
            resolved_B,
            [
                (resolved_A["4_single_source"], precision_single_any_p),
                (
                    resolved_B["4_single_source"] - resolved_A["4_single_source"],
                    precision_multi_high,
                ),
            ],
        ),
    ),
]
display(
    pl.DataFrame(
        [
            (
                label,
                r["decided"],
                r["deferred"],
                r["coverage"],
                r["precision"],
                r["overall"],
            )
            for label, r in rows
        ],
        schema=[
            "policy",
            "decided",
            "deferred",
            "coverage",
            "precision",
            "overall_correct_rate",
        ],
        orient="row",
    )
)
print(
    f"\nprecision_single_any_p = {precision_single_any_p:.4%}  (n={single_only.height:,} checkable single-candidate names, any p)"
)
print(
    f"precision_multi_high   = {precision_multi_high:.4%}  (n={multi_high.height} checkable multi-candidate names, p>=0.9)"
)

print(
    f"\nwhat's left after the rejected option ({remaining_B.height} names) -- the genuinely low-signal residual:"
)
display(
    remaining_B.join(per_name, on="pypi_name")
    .join(psm_ncand, on="pypi_name", how="left")
    .with_columns(pl.col("n_cand").fill_null(0))
    .group_by("sources", (pl.col("n_cand") > 1).alias("multi_candidate"))
    .agg(pl.len().alias("n"), pl.col("best_p").max().round(3).alias("max_best_p"))
)

policy,decided,deferred,coverage,precision,overall_correct_rate
str,i64,i64,f64,f64,f64
"""current algorithm (multiplicit…",18840,823,0.9581,0.99915,0.9573
"""Policy A (drop prob floor for …",19363,300,0.9847,0.99914,0.9839
"""(rejected) Policy A + accept m…",19644,19,0.999,0.99788,0.9969



precision_single_any_p = 99.9913%  (n=11,494 checkable single-candidate names, any p)
precision_multi_high   = 91.0828%  (n=157 checkable multi-candidate names, p>=0.9)

what's left after the rejected option (19 names) -- the genuinely low-signal residual:


sources,multi_candidate,n,max_best_p
str,bool,u32,f64
"""parselmouth""",true,16,0.776
"""conda_lock""",false,3,0.6


**Finding: Policy A wins — higher coverage, no measurable precision cost. The more
aggressive option is rejected.**

| policy | coverage | precision | overall correct rate |
|---|---|---|---|
| probability-gated (requires p ≥ 0.9 even for single-candidate answers) | 95.81% | 99.915% | 95.73% |
| **multiplicity-gated — Policy A, adopted (drops the probability floor for single-candidate answers)** | **98.47%** | **99.914%** | **98.39%** |
| rejected: Policy A + accept multi-candidate p ≥ 0.9 anyway | 99.90% | 99.788% | 99.69% |

- **Policy A dominates the probability-gated alternative.** Coverage rises from 95.8% to
  98.5% while composite precision is *unchanged* (99.915% → 99.914%, a rounding
  difference). §4a already showed why: 523 of the probability-gated alternative's 823
  deferred names are single-candidate parselmouth answers that were only being held back
  by a probability floor that doesn't actually predict anything for single-candidate
  names. There is essentially no precision/recall trade here — the floor was screening
  for the wrong risk factor, and Policy A removes it.
- **The remaining defer gap under Policy A (300 names) is dominated by the multiplicity
  bucket §4a already flagged as risky.** 281 of the 300 are high-probability (≥ 0.9)
  parselmouth answers with 2+ candidates — the same bucket that disagrees with conda-lock
  8.9% of the time where checkable (91.08% agreement). Accepting them anyway (the
  rejected option) closes coverage to 99.9% but drops composite precision to 99.79% — a
  10x increase in error, concentrated entirely in this one bucket.
  **Recommendation: reject it.** If that coverage is ever needed, surface the bucket as
  "low-confidence, needs review" rather than folding it into the same trust tier as
  everything else.
- **The last ~2% (19 names) isn't worth chasing with this data even under the rejected
  option.** 16 are multi-candidate parselmouth answers *below* 0.9 (both risk factors at
  once, max probability 0.776) and 3 are conda-lock's own ambiguous (0.6) tier with no
  second source to corroborate against. Closing this sliver requires new evidence (a
  fourth mapper, or manual curation), not a threshold change.
- **Adopted: Policy A.** §5 summarizes the resulting algorithm and its numbers.

---
## 5. Summary: the adopted algorithm

Resolution order, first rule to fire wins. Rule 4 uses **Policy A** (§4a/§4b): drop the
probability floor entirely for single-candidate parselmouth answers, since single-
candidacy alone — not raw probability — is what predicts agreement.

1. **grayskull hit** → curated, take it. (§2A: 0/84 disagreements with conda-lock)
2. **conda-lock static override (p = 1.0)** → curated, take it. (§2A: 0/17 disagreements
   with grayskull where checkable)
3. **≥ 2 mappers back the same conda name** → take it (a vote over *all* candidates each
   mapper returned, not just its top pick). (§2B/§3: 99.86–99.87% agreement where checkable)
4. **single remaining source** → take it if: non-parselmouth and p ≥ 0.9, **or**
   parselmouth and exactly one candidate at any probability. (§4a/§4b: multiplicity, not
   probability, is the real risk signal — gating on it alone beats gating on probability too)
5. otherwise → **defer** to the caller's identity/manual-review policy. Multi-candidate
   parselmouth answers at p ≥ 0.9 (281 names) are deliberately left here rather than
   folded into rule 4 — §4b showed that bucket alone carries a ~9% error rate.

In [13]:
# The final yield table, under Policy A (resolved_A and composite() are from §4b).
total_A = sum(resolved_A.values())
display(
    pl.DataFrame(
        [(k, v, round(v / total_A * 100, 2)) for k, v in resolved_A.items()],
        schema=["rule", "names", "pct"],
        orient="row",
    )
)

rule,names,pct
str,i64,f64
"""1_grayskull""",136,0.69
"""2_lock_static""",14,0.07
"""3_two_source_vote""",11652,59.26
"""4_single_source""",7561,38.45
"""5_deferred""",300,1.53


In [14]:
# Final composite precision / coverage / overall correct-answer rate, under Policy A.
final = composite(resolved_A, [(resolved_A["4_single_source"], precision_single_any_p)])
print(
    f"decided: {final['decided']:,} / total covered: {final['decided'] + final['deferred']:,}  (coverage = {final['coverage']:.2%})"
)
print(f"composite precision (of names given an answer): {final['precision']:.4%}")
print(
    f"deferred: {final['deferred']} names ({final['deferred'] / (final['decided'] + final['deferred']):.2%}) -- no answer given, neither correct nor incorrect"
)
print(
    f"\n==> estimated overall correct-answer rate across ALL {final['decided'] + final['deferred']:,} covered names: {final['overall']:.4%}"
)
print(
    "    (= composite precision x coverage; deferred names are excluded from the numerator, not counted as wrong)"
)

decided: 19,363 / total covered: 19,663  (coverage = 98.47%)
composite precision (of names given an answer): 99.9140%
deferred: 300 names (1.53%) -- no answer given, neither correct nor incorrect

==> estimated overall correct-answer rate across ALL 19,663 covered names: 98.3900%
    (= composite precision x coverage; deferred names are excluded from the numerator, not counted as wrong)


**Finding — putting it all together:**

| # | rule | names | share | precision proxy |
|---|---|---|---|---|
| 1 | grayskull hit | 136 | 0.7% | 100% (§2A) |
| 2 | conda-lock static override | 14 | 0.1% | 100% (§2A) |
| 3 | ≥ 2 mappers agree | 11,652 | 59.3% | 99.86% (§2B/§3) |
| 4 | single source (Policy A: any p if single-candidate) | 7,561 | 38.5% | 99.99% (§4a/§4b) |
| 5 | defer | 300 | 1.5% | n/a — no answer given |

- **Coverage: 98.5%** of the 19,663-name universe resolves through rules 1–4 —
  up from 95.8% for the probability-gated rule 4 (§4a/§4b), because Policy A stops
  screening single-candidate parselmouth answers on a probability score that (§3, §4a)
  never predicted agreement in the first place.
- **Composite precision (of names given an answer): ~99.9%**, essentially unchanged
  from the more conservative rule. Nearly all of the residual 0.1% risk sits in rule 3
  (the vote), which is also the rule with 60% of the volume — it's the only place worth
  spending further validation effort.
- **Estimated overall correct-answer rate across the whole covered universe:
  composite precision × coverage ≈ 99.9% × 98.5% ≈ 98.4%.** The remaining 1.5% is
  deferred, not wrong — mostly the multiplicity-risk bucket (§4b) that a caller can
  choose to surface as low-confidence rather than silently accept.

**How each section built to this:**

- **§1 (coverage):** parselmouth is the coverage backbone (19,611/19,663 names);
  conda-lock is nearly a strict subset; grayskull adds no unique names.
- **§2 (curated agreement):** the two curated sources never disagree (0/84, 0/17
  within the static tier); conda-lock's non-static high tier independently checks out
  at 99.87% against parselmouth.
- **§3 (does probability predict agreement?):** no — every confidence tier on both
  mappers agrees at 98–100%, except conda-lock's own lowest tier (63.6%, n=11).
- **§4a/§4b (multiplicity vs. probability — two candidate designs for rule 4):**
  multiplicity is the real risk signal, not probability. Gating on candidate count
  alone (Policy A) beats gating on probability-plus-candidate-count (the probability-gated
  alternative) — same precision, 2.7 points more coverage. A more aggressive option that
  also accepts multi-candidate p ≥ 0.9 answers was tested and rejected: it buys the last
  1.4 points of coverage at a 10x increase in error, concentrated entirely in that one
  bucket.

**Caveats:** these are proxy estimates, not audited ground truth. Rules 1–2 are checked
only against each other (both curated, so agreement doesn't rule out both being wrong
about the same name); rule 3's precision can't be checked at all with only two reachable
independent sources per name — 2-mapper agreement *is* the best evidence this dataset can
produce, not an external confirmation of it. All figures are a point-in-time snapshot
(`data/name_mapper_candidates.parquet`) against a parselmouth relations table that
regenerates upstream hourly — re-run §1's cache-building cell (deleting the cache) to
refresh before relying on exact figures for a production threshold change.